# Notebook 9 — Mode grimpeur

## Concept

L'utilisateur spécifie un **budget D+ cible** pour la sortie (ex: 800m sur 60km).
L'algo cherche un itinéraire qui :
- Va de A à B
- Cumule **approximativement** le D+ demandé (±15%)
- Maximise le score plaisir

## Approche V1 : génération de candidats + sélection

1. Calcule l'itinéraire baseline (sans contrainte D+)
2. Génère N variantes en privilégiant des zones vallonnées
3. Garde celle dont le D+ est le plus proche du budget

C'est une heuristique simple mais efficace en pratique.


## 1. Setup

In [7]:
import pandas as pd
import numpy as np
import networkx as nx
import time
import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine, text
from tqdm.auto import tqdm

import folium
from shapely import wkt

print(f"networkx {nx.__version__}")

networkx 3.6.1


In [8]:
DB_CONFIG = {
    "user": "postgres", "password": "4421",
    "host": "localhost", "port": 5432, "database": "velo_club",
}
url = (f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
       f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")
engine = create_engine(url, pool_pre_ping=True)

# Vérif que Notebook 3bis a bien tourné
with engine.connect() as conn:
    stats = pd.read_sql(text("""
        SELECT
            (SELECT COUNT(*) FROM osm_edges WHERE is_routable) AS routable,
            (SELECT COUNT(*) FROM edge_scores WHERE cost_factor_club > 0) AS with_cost
    """), conn)
print(stats.to_string(index=False))

 routable  with_cost
   407487     831949


In [9]:
print("Extraction depuis PostGIS...")
t0 = time.time()

query = text("""
    SELECT
        e.edge_id, e.osm_way_id, e.length_m, e.highway,
        ROUND(ST_X(ST_StartPoint(e.geom))::numeric, 6) AS u_lon,
        ROUND(ST_Y(ST_StartPoint(e.geom))::numeric, 6) AS u_lat,
        ROUND(ST_X(ST_EndPoint(e.geom))::numeric, 6) AS v_lon,
        ROUND(ST_Y(ST_EndPoint(e.geom))::numeric, 6) AS v_lat,
        COALESCE(es.cost_factor_v2_club, es.cost_factor_club, 1.0) AS cost_factor_club,
        COALESCE(es.cost_factor_v2_solo, es.cost_factor_solo, 1.0) AS cost_factor_solo,
        COALESCE(es.score_club, 0)   AS score_club,
        COALESCE(es.score_final_club, 0) AS score_final_club,
        COALESCE(es.score_final_solo, 0) AS score_final_solo
    FROM osm_edges e
    LEFT JOIN edge_scores es ON es.edge_id = e.edge_id
    WHERE e.is_routable = TRUE
      AND e.geom IS NOT NULL
      AND e.length_m > 0
""")

with engine.connect() as conn:
    df_edges = pd.read_sql(query, conn)
print(f"{len(df_edges):,} arêtes en {time.time()-t0:.0f}s")

Extraction depuis PostGIS...
407,434 arêtes en 5s


In [10]:
# IDs de nœuds
df_edges["u_id"] = df_edges["u_lat"].astype(str) + "_" + df_edges["u_lon"].astype(str)
df_edges["v_id"] = df_edges["v_lat"].astype(str) + "_" + df_edges["v_lon"].astype(str)

nodes_u = df_edges[["u_id", "u_lat", "u_lon"]].rename(
    columns={"u_id": "node_id", "u_lat": "lat", "u_lon": "lon"})
nodes_v = df_edges[["v_id", "v_lat", "v_lon"]].rename(
    columns={"v_id": "node_id", "v_lat": "lat", "v_lon": "lon"})
nodes = pd.concat([nodes_u, nodes_v]).drop_duplicates("node_id").reset_index(drop=True)

print(f"Nœuds uniques : {len(nodes):,}")
print(f"Degré moyen    : {2 * len(df_edges) / len(nodes):.2f}")

Nœuds uniques : 542,407
Degré moyen    : 1.50


In [11]:
print("Construction du graphe...")
t0 = time.time()

G = nx.MultiDiGraph()

# Nœuds avec coordonnées
node_attrs = {r.node_id: {"lat": float(r.lat), "lon": float(r.lon)}
              for r in nodes.itertuples()}
G.add_nodes_from(node_attrs.items())

# Arêtes bidirectionnelles (par défaut vélo)
edge_data = []
for r in df_edges.itertuples():
    attrs = {
        "edge_id":      int(r.edge_id),
        "osm_way_id":   int(r.osm_way_id),
        "length_m":     float(r.length_m),
        "highway":      r.highway,
        "cost_factor_club": float(r.cost_factor_club),
        "cost_factor_solo": float(r.cost_factor_solo),
        "score_club":         float(r.score_club),
        "score_final_club":   float(r.score_final_club),
        "score_final_solo":   float(r.score_final_solo),
    }
    edge_data.append((r.u_id, r.v_id, attrs))
    edge_data.append((r.v_id, r.u_id, attrs))

G.add_edges_from(edge_data)
print(f"✓ Graphe construit en {time.time()-t0:.0f}s")
print(f"  Nœuds  : {G.number_of_nodes():,}")
print(f"  Arêtes : {G.number_of_edges():,}")

Construction du graphe...
✓ Graphe construit en 7s
  Nœuds  : 542,407
  Arêtes : 814,868


In [12]:
# Plus grande composante connexe (faible)
t0 = time.time()
largest_cc = max(nx.weakly_connected_components(G), key=len)
n_before = G.number_of_nodes()
G = G.subgraph(largest_cc).copy()
print(f"Plus grande CC : {G.number_of_nodes():,} / {n_before:,} "
      f"({100*G.number_of_nodes()/n_before:.1f}%) "
      f"en {time.time()-t0:.0f}s")

Plus grande CC : 90,762 / 542,407 (16.7%) en 3s


In [13]:
def make_cost_fn(profile: str = "club_road", alpha: float = 0.5):
    assert profile in ("club_road", "solo_casual")
    key = f"cost_factor_{profile.replace('_road','').replace('_casual','')}"
    
    def cost(u, v, d):
        # MultiDiGraph : d peut être dict de dicts
        if isinstance(d, dict) and any(isinstance(x, dict) for x in d.values()):
            return min(_edge_cost(attrs, key, alpha) for attrs in d.values())
        return _edge_cost(d, key, alpha)
    return cost


def _edge_cost(attrs, key, alpha):
    L = attrs["length_m"]
    cf = attrs[key]
    return L * (cf ** alpha)


# Test
test = {"length_m": 100, "cost_factor_club": 3.0, "cost_factor_solo": 2.0}
print(f"Arête test : 100m, cost_factor_club=3.0, cost_factor_solo=2.0")
for alpha in [0.0, 0.3, 0.5, 0.8, 1.0]:
    c = _edge_cost(test, "cost_factor_club", alpha)
    print(f"  club_road, α={alpha} → coût = {c:.0f}")

Arête test : 100m, cost_factor_club=3.0, cost_factor_solo=2.0
  club_road, α=0.0 → coût = 100
  club_road, α=0.3 → coût = 139
  club_road, α=0.5 → coût = 173
  club_road, α=0.8 → coût = 241
  club_road, α=1.0 → coût = 300


In [14]:
# Snap to nearest node
def snap_to_node(G, lat, lon):
    """Trouve le nœud le plus proche."""
    coords = np.array([(d["lat"], d["lon"]) for _, d in G.nodes(data=True)])
    ids = list(G.nodes())
    lat_rad = np.radians(lat)
    dx = (coords[:, 1] - lon) * 111320 * np.cos(lat_rad)
    dy = (coords[:, 0] - lat) * 111320
    d2 = dx*dx + dy*dy
    i = int(np.argmin(d2))
    return ids[i], float(np.sqrt(d2[i]))


def route(G, start_latlon, end_latlon, profile="club_road", alpha=0.5):
    u, du = snap_to_node(G, *start_latlon)
    v, dv = snap_to_node(G, *end_latlon)
    cost_fn = make_cost_fn(profile, alpha)
    
    try:
        path = nx.dijkstra_path(G, u, v, weight=cost_fn)
    except nx.NetworkXNoPath:
        return None
    
    total_L = 0.0
    weighted_score = 0.0
    edges = []
    coords = [(G.nodes[path[0]]["lat"], G.nodes[path[0]]["lon"])]
    score_key = f"score_final_{profile.replace('_road','').replace('_casual','')}"
    cost_key = f"cost_factor_{profile.replace('_road','').replace('_casual','')}"
    
    for i in range(len(path) - 1):
        a, b = path[i], path[i+1]
        cands = G[a][b]
        best = min(cands.keys(), key=lambda k: _edge_cost(cands[k], cost_key, alpha))
        attrs = cands[best]
        edges.append(attrs)
        total_L += attrs["length_m"]
        weighted_score += attrs["length_m"] * attrs[score_key]
        coords.append((G.nodes[b]["lat"], G.nodes[b]["lon"]))
    
    return {
        "path": path, "edges": edges, "coords": coords,
        "total_length_m": total_L,
        "mean_score": weighted_score / max(total_L, 1),
        "profile": profile, "alpha": alpha,
        "snap_m": (du, dv),
    }

print("route() prête")

route() prête


In [15]:
# Continuer depuis l'environnement du Notebook 4 ou 6
# G doit être chargé, route() doit être défini
import pandas as pd
import numpy as np
from sqlalchemy import text

# Récupération du D+ par arête depuis la DB
print("Récupération du D+ par arête...")

with engine.connect() as conn:
    d_plus_df = pd.read_sql(text("""
        SELECT edge_id, d_plus_m FROM osm_edges WHERE d_plus_m IS NOT NULL;
    """), conn)
d_plus_map = dict(zip(d_plus_df["edge_id"], d_plus_df["d_plus_m"]))
print(f"{len(d_plus_map):,} arêtes avec D+")

# Inject dans le graphe G existant
for u, v, attrs in G.edges(data=True):
    eid = attrs.get("edge_id")
    if eid in d_plus_map:
        attrs["d_plus_m"] = float(d_plus_map[eid])
    else:
        attrs["d_plus_m"] = 0.0
print("D+ injecté dans le graphe")

Récupération du D+ par arête...
506,169 arêtes avec D+
D+ injecté dans le graphe


In [17]:
POINTS = {
    "pantin":         (48.8922, 2.4014),
    "chevreuse":      (48.7076, 2.0387),
    "fontainebleau":  (48.4020, 2.7015),
    "cergy":          (49.0390, 2.0768),
    "rambouillet":    (48.6447, 1.8285),
    "meaux":          (48.9609, 2.8783),
}

# Test 1 : Pantin → Chevreuse, profil club, différents alpha
print("=" * 70)
print("Pantin → Chevreuse — profil club_road, variations alpha")
print("=" * 70)
results_club = {}
for a in [0.0, 0.3, 0.5, 10]:
    t0 = time.time()
    r = route(G, POINTS["pantin"], POINTS["chevreuse"], profile="club_road", alpha=a)
    if r:
        results_club[a] = r
        print(f"α = {a} | {r['total_length_m']/1000:.1f} km | "
              f"score moy {r['mean_score']:.3f} | "
              f"{time.time()-t0:.1f}s")

Pantin → Chevreuse — profil club_road, variations alpha
α = 0.0 | 53.5 km | score moy 0.428 | 0.5s
α = 0.3 | 53.7 km | score moy 0.425 | 0.7s
α = 0.5 | 54.2 km | score moy 0.444 | 0.5s
α = 10 | 79.8 km | score moy 0.428 | 0.6s


## 2. Fonction de routing grimpeur

In [18]:
def compute_d_plus(result):
    """Calcule le D+ total d'un résultat de route()."""
    if not result or "edges" not in result:
        return 0
    return sum(e.get("d_plus_m", 0) for e in result["edges"])


def climber_route(G, A, B, d_plus_target, d_plus_tolerance=0.20,
                  profile="club_road", alpha_range=(0.5, 2.0), n_candidates=8):
    """
    Cherche un itinéraire respectant un budget D+ cible.
    
    Args:
        d_plus_target: D+ cible en mètres (ex: 800)
        d_plus_tolerance: tolérance relative (0.20 = ±20%)
        alpha_range: gamme d'alpha à tester
        n_candidates: nombre de variantes à générer
    
    Returns:
        Le meilleur itinéraire avec son D+ et son score
    """
    candidates = []
    
    alphas = np.linspace(alpha_range[0], alpha_range[1], n_candidates)
    for a in alphas:
        r = route(G, A, B, profile=profile, alpha=a)
        if r is None:
            continue
        d_plus = compute_d_plus(r)
        # Score combiné : on veut maximiser le score plaisir
        # ET être proche du d_plus_target
        d_plus_match = 1 - min(1, abs(d_plus - d_plus_target) / d_plus_target)
        combined = 0.5 * r["mean_score"] + 0.5 * d_plus_match
        
        candidates.append({
            "alpha": a,
            "distance_km": r["total_length_m"] / 1000,
            "d_plus_m": d_plus,
            "score": r["mean_score"],
            "match_d_plus": d_plus_match,
            "combined": combined,
            "result": r,
        })
    
    if not candidates:
        return None
    
    # Filtre les candidats dans la tolérance
    d_min = d_plus_target * (1 - d_plus_tolerance)
    d_max = d_plus_target * (1 + d_plus_tolerance)
    in_range = [c for c in candidates if d_min <= c["d_plus_m"] <= d_max]
    
    if in_range:
        # Meilleur score parmi ceux dans la fourchette
        best = max(in_range, key=lambda c: c["score"])
        best["status"] = "in_range"
    else:
        # Aucun dans la fourchette, on prend le plus proche du target
        best = max(candidates, key=lambda c: c["combined"])
        best["status"] = "approximate"
    
    return best


# Test
print("Test : Pantin → Chevreuse, budget D+ = 800m")
result = climber_route(G, POINTS["pantin"], POINTS["chevreuse"],
                       d_plus_target=800, d_plus_tolerance=0.20,
                       profile="club_road", n_candidates=6)
if result:
    print(f"  α={result['alpha']:.2f}")
    print(f"  Distance : {result['distance_km']:.1f} km")
    print(f"  D+ : {result['d_plus_m']:.0f} m (cible 800)")
    print(f"  Score : {result['score']:.3f}")
    print(f"  Status : {result['status']}")

Test : Pantin → Chevreuse, budget D+ = 800m
  α=0.50
  Distance : 54.2 km
  D+ : 930 m (cible 800)
  Score : 0.444
  Status : in_range


## 3. Visualisation

In [19]:
# Comparaison : itinéraire normal vs mode grimpeur
print("Comparaison normal vs grimpeur :")
print("="*60)

r_normal = route(G, POINTS["pantin"], POINTS["chevreuse"],
                profile="club_road", alpha=1.0)
d_normal = compute_d_plus(r_normal)
print(f"Normal (α=1.0) : {r_normal['total_length_m']/1000:.1f}km, "
      f"D+={d_normal:.0f}m, score {r_normal['mean_score']:.3f}")

for target in [500, 800, 1200]:
    r = climber_route(G, POINTS["pantin"], POINTS["chevreuse"],
                     d_plus_target=target, n_candidates=6)
    if r:
        print(f"Grimpeur D+={target} : {r['distance_km']:.1f}km, "
              f"D+={r['d_plus_m']:.0f}m, score {r['score']:.3f} ({r['status']})")

Comparaison normal vs grimpeur :
Normal (α=1.0) : 58.4km, D+=935m, score 0.440
Grimpeur D+=500 : 54.2km, D+=930m, score 0.444 (approximate)
Grimpeur D+=800 : 54.2km, D+=930m, score 0.444 (in_range)
Grimpeur D+=1200 : 66.0km, D+=1038m, score 0.443 (in_range)


## Notes

Le mode grimpeur V1 explore une grille d'α et choisit le meilleur. Limites :
- Cherche au sein d'un petit ensemble (pas de vraie optimisation)
- Si A→B ne contient pas de relief, impossible d'atteindre un D+ haut
- Pour augmenter D+, il faut passer par des zones vallonnées (Yvelines sud, vallée Chevreuse)

V2 : intégrer le D+ directement comme contrainte de routing (Constrained Shortest Path).
